In [ ]:
The goal in mind for the analysis is compare how news about security concerns can lead to decrease in certain python packages being used.
How I can achieve this is by using PyPI data that keeps track of all the times python packages get installed using pip and then
    look at news about security breaches in different python packages and compare the installs or uninstalls around that time with the data

In [ ]:
"""
Compare word frequency between two books (PDFs).

Usage:
    python compare_book_word_freq.py book1.pdf book2.pdf

Requires:
    pip install pypdf matplotlib
"""

import sys
import re
from collections import Counter
from pypdf import PdfReader
import matplotlib.pyplot as plt

# Common words to ignore so the chart shows meaningful words instead of
# "the", "and", "a", etc. Feel free to edit this list.
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "of", "to", "in", "on",
    "for", "with", "as", "at", "by", "from", "is", "was", "were", "are",
    "be", "been", "being", "it", "its", "this", "that", "these", "those",
    "he", "she", "they", "we", "you", "i", "his", "her", "their", "our",
    "your", "him", "them", "us", "not", "no", "so", "up", "out", "about",
    "into", "than", "then", "there", "here", "when", "what", "which",
    "who", "whom", "will", "would", "could", "should", "can", "do",
    "does", "did", "had", "has", "have", "all", "one", "said", "would",
}


def extract_text(pdf_path):
    """Pull all text out of a PDF file."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + " "
    return text


def count_words(text, remove_stopwords=True):
    """Lowercase, strip punctuation, split into words, and count them."""
    words = re.findall(r"[a-zA-Z']+", text.lower())
    if remove_stopwords:
        words = [w for w in words if w not in STOPWORDS]
    return Counter(words)


def plot_top_words(counter1, counter2, name1, name2, top_n=15):
    """Show two side-by-side bar charts of the most common words."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, counter, name in zip(axes, [counter1, counter2], [name1, name2]):
        top_words = counter.most_common(top_n)
        words = [w for w, _ in top_words]
        counts = [c for _, c in top_words]

        ax.barh(words, counts, color="steelblue")
        ax.invert_yaxis()  # highest count at the top
        ax.set_title(f"Top {top_n} Words in {name}")
        ax.set_xlabel("Frequency")

    plt.tight_layout()
    plt.savefig("word_frequency_comparison.png", dpi=150)
    print("Saved chart to word_frequency_comparison.png")
    plt.show()


def main():
    if len(sys.argv) != 3:
        print("Usage: python compare_book_word_freq.py book1.pdf book2.pdf")
        sys.exit(1)

    pdf1_path, pdf2_path = sys.argv[1], sys.argv[2]

    print(f"Reading {pdf1_path}...")
    text1 = extract_text(pdf1_path)
    print(f"Reading {pdf2_path}...")
    text2 = extract_text(pdf2_path)

    counter1 = count_words(text1)
    counter2 = count_words(text2)

    total1 = sum(counter1.values())
    total2 = sum(counter2.values())

    print(f"\n{pdf1_path}: {total1} words (after removing stopwords)")
    print(f"{pdf2_path}: {total2} words (after removing stopwords)")

    print(f"\nTop 10 words in {pdf1_path}:")
    for word, count in counter1.most_common(10):
        print(f"  {word}: {count}")

    print(f"\nTop 10 words in {pdf2_path}:")
    for word, count in counter2.most_common(10):
        print(f"  {word}: {count}")

    plot_top_words(counter1, counter2, pdf1_path, pdf2_path)


if __name__ == "__main__":
    main()